# Azure AI Search with Perplexity Contextualized Embeddings

In this notebook, I'll show you how to use Perplexity's `pplx-embed` contextualized embedding models with Azure AI Search to build a hybrid search pipeline over multiple PDF documents.

**What makes this different from OpenAI embeddings?**

Standard embedding models like OpenAI `text-embedding-3-large` embed each chunk **independently** — the model has no idea what comes before or after. Perplexity's contextualized embeddings send all chunks from a document **together**, so each vector encodes inter-chunk relationships. This is the "golden chunk" insight:

| Approach | Input | Result |
|---|---|---|
| **Standard** | `embed("chunk about attention heads")` | Vector in isolation |
| **Contextualized** | `embed(["intro", "related work", "chunk about attention heads", "experiments"])` | Vector that *knows* this chunk follows related work and precedes experiments |

A query like *"how does multi-head attention work?"* retrieves the **right** chunk even when its text alone is ambiguous, because the embedding captures document-level context.

**Pipeline overview:**

```
PDFs → Page-based chunks → Contextualized embeddings → Azure AI Search → Hybrid search
```

**Key capabilities demonstrated:**

1. **Contextualized embeddings** via `/v1/contextualizedembeddings` — chunks share document context
2. **Multi-document support** — each PDF's chunks are contextualized independently in a single API call
3. **Matryoshka Representation Learning (MRL)** — truncate embeddings to smaller dimensions (128–1024 for 0.6b, 128–2560 for 4b) while retaining quality
4. **INT8 quantized by default** (`base64_int8`) — smaller payloads out of the box
5. **Hybrid search** — BM25 text matching + vector similarity + semantic reranking

## Install Libraries

In [3]:
! pip install azure-search-documents perplexityai pypdf numpy requests python-dotenv tenacity --quiet


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## Configure Environment

Set the following environment variables in a `.env` file at the project root, or export them in your shell:

```bash
PERPLEXITY_API_KEY=pplx-...
AZURE_SEARCH_ENDPOINT=https://<your-service>.search.windows.net
AZURE_SEARCH_KEY=<your-admin-key>
```

You can get a Perplexity API key from [perplexity.ai/settings/api](https://www.perplexity.ai/settings/api). For Azure AI Search, follow the [quickstart guide](https://learn.microsoft.com/azure/search/search-create-service-portal) to create a service.

In [4]:
import os
from dotenv import load_dotenv

load_dotenv()

PERPLEXITY_API_KEY = os.environ.get("PERPLEXITY_API_KEY")
AZURE_SEARCH_ENDPOINT = os.environ.get("AZURE_SEARCH_ENDPOINT")
AZURE_SEARCH_KEY = os.environ.get("AZURE_SEARCH_KEY")

# Validate that all required environment variables are set
missing = [v for v in ["PERPLEXITY_API_KEY", "AZURE_SEARCH_ENDPOINT", "AZURE_SEARCH_KEY"] if not os.environ.get(v)]
if missing:
    raise EnvironmentError(f"Missing environment variables: {', '.join(missing)}")

print("All environment variables are set.")

All environment variables are set.


## Configure Embedding Model

Perplexity offers two contextualized embedding models:

| Model | Dimensions | MRL Range | Price |
|---|---|---|---|
| `pplx-embed-context-v1-0.6b` | 1024 | 128–1024 | $0.008/1M tokens |
| `pplx-embed-context-v1-4b` | 2560 | 128–2560 | Higher |

We'll use the 0.6b model at full 1024 dimensions. You can set `EMBED_DIMS` to a smaller value (e.g., 512 or 256) to test Matryoshka truncation.

**Other differences from OpenAI `text-embedding-3-large`:**
- INT8 quantized by default (`base64_int8`) — smaller payloads out of the box
- Embeddings are **unnormalized** — must use cosine similarity, not dot product

In [5]:
# pplx-embed-context-v1-0.6b: 1024 dims, $0.008/1M tokens (cheapest context model)
# Swap to pplx-embed-context-v1-4b for 2560 dims / higher quality
EMBED_MODEL = "pplx-embed-context-v1-0.6b"
EMBED_DIMS = 1024  # Full dims for 0.6b. Set to 512 or 256 for MRL truncation.

INDEX_NAME = "pplx-embed-demo"
MAX_CHUNK_CHARS = 4000  # Split pages longer than this into sub-chunks
BATCH_SIZE = 512  # Perplexity max: 512 documents per request

# Two arxiv papers to demonstrate multi-document contextualized embeddings
PDF_SOURCES = [
    {
        "url": "https://arxiv.org/pdf/1706.03762",
        "title": "Attention Is All You Need",
        "source_file": "1706.03762.pdf",
    },
    {
        "url": "https://arxiv.org/pdf/2404.16130",
        "title": "From Local to Global: A Graph RAG Approach to Query-Focused Summarization",
        "source_file": "2404.16130.pdf",
    },
]

print(f"Model: {EMBED_MODEL}")
print(f"Dimensions: {EMBED_DIMS}")
print(f"Index: {INDEX_NAME}")
print(f"Documents: {len(PDF_SOURCES)}")

Model: pplx-embed-context-v1-0.6b
Dimensions: 1024
Index: pplx-embed-demo
Documents: 2


## Download and Chunk the PDFs

We'll use two arxiv papers as our test documents:
1. **"Attention Is All You Need"** — the original transformer paper
2. **"From Local to Global: A Graph RAG Approach"** — Microsoft's GraphRAG paper

The chunking strategy is one chunk per page — if a page exceeds `MAX_CHUNK_CHARS`, it gets split into sub-chunks at sentence boundaries.

**Important:** Chunk **order** matters for contextualized embeddings. The model relies on sequential context, so chunks must be sent in reading order within each document.

In [6]:
import tempfile
from pathlib import Path
import requests
from pypdf import PdfReader


def download_pdf(url: str) -> Path:
    """Download a PDF to a temp file and return its path."""
    print(f"Downloading PDF from {url} ...")
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    tmp = tempfile.NamedTemporaryFile(suffix=".pdf", delete=False)
    tmp.write(resp.content)
    tmp.close()
    print(f"  Saved to {tmp.name} ({len(resp.content) / 1024:.0f} KB)")
    return Path(tmp.name)


def chunk_pdf(pdf_path: Path) -> list[dict]:
    """
    Extract text from a PDF and return ordered chunks with metadata.

    Strategy: one chunk per page. If a page exceeds MAX_CHUNK_CHARS, split it
    into sub-chunks at sentence boundaries.
    """
    reader = PdfReader(str(pdf_path))
    chunks = []

    for page_num, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        text = text.strip()
        if not text:
            continue

        if len(text) <= MAX_CHUNK_CHARS:
            chunks.append({
                "page_number": page_num,
                "chunk_index": 0,
                "text": text,
            })
        else:
            # Split long pages into sub-chunks at sentence boundaries
            sentences = text.replace("\n", " ").split(". ")
            current, idx = "", 0
            for sentence in sentences:
                candidate = (current + ". " + sentence).strip() if current else sentence
                if len(candidate) > MAX_CHUNK_CHARS and current:
                    chunks.append({
                        "page_number": page_num,
                        "chunk_index": idx,
                        "text": current.strip(),
                    })
                    idx += 1
                    current = sentence
                else:
                    current = candidate
            if current.strip():
                chunks.append({
                    "page_number": page_num,
                    "chunk_index": idx,
                    "text": current.strip(),
                })

    print(f"  Extracted {len(chunks)} chunks from {len(reader.pages)} pages")
    return chunks

In [7]:
# Download and chunk each PDF
all_docs = []
for source in PDF_SOURCES:
    print(f"\n--- {source['title']} ---")
    pdf_path = download_pdf(source["url"])
    chunks = chunk_pdf(pdf_path)
    os.unlink(pdf_path)
    all_docs.append({
        "title": source["title"],
        "source_file": source["source_file"],
        "chunks": chunks,
    })

print(f"\nTotal: {sum(len(d['chunks']) for d in all_docs)} chunks across {len(all_docs)} documents")


--- Attention Is All You Need ---
  Saved to /var/folders/wr/dct0mymx2pl9k78dhnzt1p1w0000gn/T/tmpgjrbhxac.pdf (2163 KB)
  Extracted 16 chunks from 15 pages

--- From Local to Global: A Graph RAG Approach to Query-Focused Summarization ---
  Saved to /var/folders/wr/dct0mymx2pl9k78dhnzt1p1w0000gn/T/tmpygtpl5gj.pdf (6732 KB)
  Extracted 31 chunks from 26 pages

Total: 47 chunks across 2 documents


### Preview the Chunks

Let's inspect the first few chunks from each document to verify our parsing is working correctly.

In [8]:
for doc in all_docs:
    print(f"\n{'='*60}")
    print(f"Document: {doc['title']}")
    print(f"{'='*60}")
    for i, chunk in enumerate(doc["chunks"][:2]):
        print(f"\n--- Chunk {i} (Page {chunk['page_number']}, Index {chunk['chunk_index']}) ---")
        print(chunk["text"][:300] + "..." if len(chunk["text"]) > 300 else chunk["text"])


Document: Attention Is All You Need

--- Chunk 0 (Page 1, Index 0) ---
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Par...

--- Chunk 1 (Page 2, Index 0) ---
1 Introduction Recurrent neural networks, long short-term memory [13] and gated recurrent [7] neural networks in particular, have been firmly established as state of the art approaches in sequence modeling and transduction problems such as language modeling and machine translation [ 35, 2, 5]. Numer...

Document: From Local to Global: A Graph RAG Approach to Query-Focused Summarization

--- Chunk 0 (Page 1, Index 0) ---
From Local to Global: A GraphRAG Approach to
Query-Focused Summarization
Darren Edge1† Ha Trinh1† Newman Cheng2 Joshua Bradley2 Alex Chao3
Apurva Mody3 Steven Truitt

## Generate Contextualized Embeddings

This is the core differentiator. Instead of embedding each chunk independently, we send **all chunks from each document together** as a nested array. The model sees neighboring chunks, so each vector encodes inter-chunk relationships.

**Multi-document API input format:**
```python
# Nested arrays: outer = documents, inner = chunks in reading order
# Each document's chunks share context with each other, but NOT across documents
input = [
    [doc1_chunk1, doc1_chunk2, doc1_chunk3],  # Transformer paper chunks
    [doc2_chunk1, doc2_chunk2],                # GraphRAG paper chunks
]
```

This is the key architectural detail — chunks within the same inner array share context, but the Transformer paper's chunks don't "see" the GraphRAG paper's chunks (and vice versa). This is correct behavior since they are separate documents.

**API limits:** 512 documents, 16,000 total chunks, 120K tokens per request.

**Free tier rate limits:** The free tier has both a request-per-minute limit (20 RPM) *and* a token-per-minute limit. For larger documents, we batch chunks into groups of ~15 per API call with 60-second pauses between batches to stay within limits.

**Decoding:** Perplexity returns embeddings as base64-encoded signed INT8 values by default. We decode these to float32 for Azure AI Search (which expects `Collection(Edm.Single)`).

In [9]:
import base64
import math
import time

import numpy as np
from perplexity import Perplexity
from tenacity import retry, wait_fixed, stop_after_attempt, before_sleep_log
import logging

logger = logging.getLogger("tenacity")
logger.setLevel(logging.INFO)

# Batching config for free tier token-per-minute limits
CHUNKS_PER_BATCH = 15  # ~11K tokens per batch — safe for free tier TPM limit
BATCH_PAUSE_SECS = 60  # Pause between batches to let the TPM bucket refill


def decode_base64_int8(b64_string: str) -> list[float]:
    """
    Decode a Perplexity base64_int8 embedding to a list of floats.

    The int8 values are in [-128, 127]. We convert to float32 directly.
    Cosine similarity handles the unnormalized magnitudes correctly.
    """
    raw_bytes = base64.b64decode(b64_string)
    int8_array = np.frombuffer(raw_bytes, dtype=np.int8)
    return int8_array.astype(np.float32).tolist()


@retry(wait=wait_fixed(60), stop=stop_after_attempt(3), before_sleep=before_sleep_log(logger, logging.INFO))
def _call_contextualized_embeddings(client, nested_input):
    """Call the Perplexity API with fixed 60s retry on rate limits."""
    return client.contextualized_embeddings.create(
        input=nested_input,
        model=EMBED_MODEL,
        dimensions=EMBED_DIMS,
    )


def generate_embeddings_multi_doc(client: Perplexity, docs: list[dict]) -> list[dict]:
    """
    Generate contextualized embeddings for each document with token-aware batching.

    For documents with many chunks, we split into batches of CHUNKS_PER_BATCH
    to stay within the free tier's token-per-minute limit. Each batch is sent
    as a separate API call with BATCH_PAUSE_SECS pause between calls.
    """
    for i, doc in enumerate(docs):
        texts = [c["text"] for c in doc["chunks"]]
        num_batches = math.ceil(len(texts) / CHUNKS_PER_BATCH)
        print(f"  Embedding '{doc['title']}' ({len(texts)} chunks in {num_batches} batch(es)) ...")

        all_vectors = []
        for batch_idx in range(num_batches):
            start = batch_idx * CHUNKS_PER_BATCH
            end = min(start + CHUNKS_PER_BATCH, len(texts))
            batch_texts = texts[start:end]

            print(f"    Batch {batch_idx + 1}/{num_batches}: chunks {start}-{end - 1} ({len(batch_texts)} chunks)")
            response = _call_contextualized_embeddings(client, [batch_texts])

            batch_vectors = [
                decode_base64_int8(chunk_emb.embedding)
                for chunk_emb in response.data[0].data
            ]
            all_vectors.extend(batch_vectors)

            # Pause between batches to stay within TPM limit
            if batch_idx < num_batches - 1:
                print(f"    Pausing {BATCH_PAUSE_SECS}s for rate limit ...")
                time.sleep(BATCH_PAUSE_SECS)

        doc["vectors"] = all_vectors
        print(f"    -> {len(doc['vectors'])} vectors, {EMBED_DIMS} dims each")

        # Pause between documents
        if i < len(docs) - 1:
            print(f"    Pausing {BATCH_PAUSE_SECS}s between documents ...")
            time.sleep(BATCH_PAUSE_SECS)

    total = sum(len(d["vectors"]) for d in docs)
    print(f"\n  Total: {total} vectors across {len(docs)} documents")
    return docs


@retry(wait=wait_fixed(60), stop=stop_after_attempt(3), before_sleep=before_sleep_log(logger, logging.INFO))
def embed_query(client: Perplexity, query: str) -> list[float]:
    """
    Embed a search query using the same contextualized model.

    Queries are wrapped as [[query]] — a single-chunk, single-document input.
    This ensures query and document embeddings live in the same vector space.
    """
    response = client.contextualized_embeddings.create(
        input=[[query]],
        model=EMBED_MODEL,
        dimensions=EMBED_DIMS,
    )
    return decode_base64_int8(response.data[0].data[0].embedding)

In [10]:
import json

EMBEDDINGS_CACHE = "pplx_embeddings.json"

pplx_client = Perplexity(api_key=PERPLEXITY_API_KEY)

# Check for cached embeddings to skip the slow API calls on re-runs
if os.path.exists(EMBEDDINGS_CACHE):
    print(f"Loading cached embeddings from {EMBEDDINGS_CACHE} ...")
    with open(EMBEDDINGS_CACHE) as f:
        cached = json.load(f)
    # Attach vectors back to the docs
    for doc in all_docs:
        doc_vectors = [
            item["vector"]
            for item in cached
            if item["source_file"] == doc["source_file"]
        ]
        doc["vectors"] = doc_vectors
        print(f"  {doc['title']}: {len(doc_vectors)} vectors loaded")
    print(f"Total: {sum(len(d['vectors']) for d in all_docs)} vectors from cache")
else:
    print("No cache found — generating embeddings via API (this takes a few minutes on free tier) ...")
    all_docs = generate_embeddings_multi_doc(pplx_client, all_docs)

    # Save to cache for future re-runs
    cache_data = []
    for doc in all_docs:
        for chunk, vector in zip(doc["chunks"], doc["vectors"]):
            cache_data.append({
                "title": doc["title"],
                "source_file": doc["source_file"],
                "page_number": chunk["page_number"],
                "chunk_index": chunk["chunk_index"],
                "text": chunk["text"],
                "vector": vector,
            })
    with open(EMBEDDINGS_CACHE, "w") as f:
        json.dump(cache_data, f)
    print(f"\nSaved {len(cache_data)} embeddings to {EMBEDDINGS_CACHE}")

Loading cached embeddings from pplx_embeddings.json ...
  Attention Is All You Need: 16 vectors loaded
  From Local to Global: A Graph RAG Approach to Query-Focused Summarization: 31 vectors loaded
Total: 47 vectors from cache


### Verify the Embeddings

Let's inspect the shape and a sample vector to confirm everything looks correct.

In [11]:
for doc in all_docs:
    vectors = doc["vectors"]
    print(f"\n{doc['title']}:")
    print(f"  Vectors: {len(vectors)}")
    print(f"  Dimensions: {len(vectors[0])}")
    print(f"  Sample (first 10 dims): {vectors[0][:10]}")
    print(f"  Value range: [{min(vectors[0]):.1f}, {max(vectors[0]):.1f}]")


Attention Is All You Need:
  Vectors: 16
  Dimensions: 1024
  Sample (first 10 dims): [-6.0, -37.0, 0.0, -30.0, -6.0, 62.0, 2.0, 24.0, -55.0, 3.0]
  Value range: [-60.0, 62.0]

From Local to Global: A Graph RAG Approach to Query-Focused Summarization:
  Vectors: 31
  Dimensions: 1024
  Sample (first 10 dims): [-5.0, -40.0, 0.0, -10.0, 1.0, 35.0, 7.0, 37.0, -36.0, -22.0]
  Value range: [-67.0, 65.0]


## Create the Azure AI Search Index

We'll create an index optimized for Perplexity embeddings. Key design decisions:

- **COSINE metric on HNSW** — Required because pplx embeddings are **unnormalized**. Using `dotProduct` would give wrong results (unlike OpenAI which pre-normalizes).
- **Semantic ranker config** — Enables hybrid retrieval (text + vector + reranking).
- **No Azure OpenAI vectorizer** — We embed client-side with Perplexity.

For more details on creating vector indexes, see the [Azure AI Search documentation](https://learn.microsoft.com/azure/search/vector-search-how-to-create-index).

In [12]:
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex,
    SearchField,
    SearchFieldDataType,
    SimpleField,
    SearchableField,
    VectorSearch,
    HnswAlgorithmConfiguration,
    HnswParameters,
    VectorSearchProfile,
    SemanticConfiguration,
    SemanticSearch,
    SemanticPrioritizedFields,
    SemanticField,
)

credential = AzureKeyCredential(AZURE_SEARCH_KEY)
index_client = SearchIndexClient(endpoint=AZURE_SEARCH_ENDPOINT, credential=credential)

In [13]:
fields = [
    SimpleField(
        name="id",
        type=SearchFieldDataType.String,
        key=True,
        filterable=True,
    ),
    SearchableField(
        name="content",
        type=SearchFieldDataType.String,
        analyzer_name="en.microsoft",
    ),
    SearchField(
        name="content_vector",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True,
        vector_search_dimensions=EMBED_DIMS,
        vector_search_profile_name="pplx-vector-profile",
    ),
    SimpleField(
        name="page_number",
        type=SearchFieldDataType.Int32,
        filterable=True,
        sortable=True,
    ),
    SimpleField(
        name="chunk_index",
        type=SearchFieldDataType.Int32,
        filterable=True,
        sortable=True,
    ),
    SimpleField(
        name="document_title",
        type=SearchFieldDataType.String,
        filterable=True,
    ),
    SimpleField(
        name="source_file",
        type=SearchFieldDataType.String,
        filterable=True,
    ),
]

# HNSW with COSINE — the only correct metric for unnormalized int8 embeddings
vector_search = VectorSearch(
    algorithms=[
        HnswAlgorithmConfiguration(
            name="pplx-hnsw",
            parameters=HnswParameters(
                m=4,
                ef_construction=400,
                ef_search=500,
                metric="cosine",
            ),
        ),
    ],
    profiles=[
        VectorSearchProfile(
            name="pplx-vector-profile",
            algorithm_configuration_name="pplx-hnsw",
        ),
    ],
)

# Semantic ranker for hybrid retrieval: text matching + vector + reranker
semantic_config = SemanticConfiguration(
    name="pplx-semantic-ranker-config",
    prioritized_fields=SemanticPrioritizedFields(
        content_fields=[SemanticField(field_name="content")],
    ),
)
semantic_search = SemanticSearch(configurations=[semantic_config])

index = SearchIndex(
    name=INDEX_NAME,
    fields=fields,
    vector_search=vector_search,
    semantic_search=semantic_search,
)

index_client.create_or_update_index(index)
print(f"Index '{INDEX_NAME}' created/updated")

Index 'pplx-embed-demo' created/updated


## Upload Documents to Azure AI Search

Now we'll upload the chunks along with their contextualized embeddings. Each document includes:
- `content` — the raw text for BM25 keyword matching
- `content_vector` — the contextualized embedding for vector search
- Metadata fields (`page_number`, `chunk_index`, `document_title`, `source_file`) for filtering and display

In [14]:
search_client = SearchClient(
    endpoint=AZURE_SEARCH_ENDPOINT,
    index_name=INDEX_NAME,
    credential=credential,
)

# Flatten all documents into search documents
documents = []
for doc in all_docs:
    for chunk, vector in zip(doc["chunks"], doc["vectors"]):
        doc_id = f"{doc['source_file'].replace('.', '_')}_{chunk['page_number']}_{chunk['chunk_index']}"
        documents.append({
            "id": doc_id,
            "content": chunk["text"],
            "content_vector": vector,
            "page_number": chunk["page_number"],
            "chunk_index": chunk["chunk_index"],
            "document_title": doc["title"],
            "source_file": doc["source_file"],
        })

# Upload in batches of 1000 (Azure AI Search limit per request)
batch_size = 1000
for i in range(0, len(documents), batch_size):
    batch = documents[i : i + batch_size]
    result = search_client.upload_documents(documents=batch)
    succeeded = sum(1 for r in result if r.succeeded)
    print(f"Uploaded {succeeded}/{len(batch)} documents (batch {i // batch_size + 1})")

print(f"\nTotal documents in index: {len(documents)}")

Uploaded 47/47 documents (batch 1)

Total documents in index: 47


## Run Hybrid Search Queries

Now for the payoff. We'll run hybrid search combining three signals:

1. **BM25 keyword matching** on the `content` field
2. **Cosine similarity** against the contextualized embedding
3. **Microsoft's semantic reranker** for final relevance scoring

The contextualized embeddings shine here — the vector search finds chunks whose **document-level context** matches the query intent, not just surface-level keyword overlap. With two papers indexed, we can see how the search correctly routes queries to the right document.

In [15]:
import time
from azure.search.documents.models import VectorizedQuery


def hybrid_search(search_client: SearchClient, query_vector: list[float], query_text: str):
    """Execute hybrid search: full-text BM25 + vector similarity + semantic reranking."""
    vector_query = VectorizedQuery(
        vector=query_vector,
        k_nearest_neighbors=5,
        fields="content_vector",
    )

    results = search_client.search(
        search_text=query_text,
        vector_queries=[vector_query],
        query_type="semantic",
        semantic_configuration_name="pplx-semantic-ranker-config",
        top=5,
        select=["id", "content", "page_number", "chunk_index", "document_title"],
    )

    print(f"\n{'='*80}")
    print(f"QUERY: {query_text}")
    print(f"{'='*80}")

    for i, result in enumerate(results, 1):
        score = result.get("@search.score", 0)
        reranker_score = result.get("@search.reranker_score", None)
        page = result["page_number"]
        title = result["document_title"]
        content_preview = result["content"][:200].replace("\n", " ")

        print(f"\n  [{i}] {title} — Page {page} | Score: {score:.4f}", end="")
        if reranker_score is not None:
            print(f" | Reranker: {reranker_score:.4f}", end="")
        print(f"\n      {content_preview}...")

In [16]:
# Wait a few seconds for the index to commit documents
print("Waiting 3s for index to commit documents...")
time.sleep(3)

queries = [
    # Should match the Transformer paper
    "How does multi-head attention work?",
    # Should match the GraphRAG paper
    "How does GraphRAG handle community detection for summarization?",
    # Could match both — tests cross-document retrieval
    "What are the key innovations compared to previous approaches?",
]

for i, query in enumerate(queries):
    query_vector = embed_query(pplx_client, query)
    hybrid_search(search_client, query_vector, query)
    if i < len(queries) - 1:
        time.sleep(5)  # Pause between queries to stay within 20 RPM

Waiting 3s for index to commit documents...

QUERY: How does multi-head attention work?

  [1] Attention Is All You Need — Page 5 | Score: 0.0331 | Reranker: 3.2109
      output values. These are concatenated and once again projected, resulting in the final values, as depicted in Figure 2. Multi-head attention allows the model to jointly attend to information from diff...

  [2] Attention Is All You Need — Page 4 | Score: 0.0331 | Reranker: 3.1790
      Scaled Dot-Product Attention  Multi-Head Attention Figure 2: (left) Scaled Dot-Product Attention. (right) Multi-Head Attention consists of several attention layers running in parallel. of the values, ...

  [3] Attention Is All You Need — Page 3 | Score: 0.0297 | Reranker: 2.7706
      Figure 1: The Transformer - model architecture. The Transformer follows this overall architecture using stacked self-attention and point-wise, fully connected layers for both the encoder and decoder, ...

  [4] Attention Is All You Need — Page 1 | Score: 

## Comparison: Perplexity pplx-embed vs. OpenAI text-embedding-3-large

| Feature | Perplexity pplx-embed | OpenAI text-embedding-3-large |
|---|---|---|
| Contextualized embeddings | **YES** — chunks share doc context | NO — each chunk independent |
| Multi-document in one call | **YES** — nested arrays per document | NO — flat array of strings |
| MRL (flexible dimensions) | YES — 128 to 2560 | YES — 256 to 3072 |
| Default quantization | INT8 (`base64_int8`) | float32 |
| Normalization | **UNNORMALIZED** (use cosine) | Pre-normalized (dot product OK) |
| Binary quantization | Built-in (`base64_binary`) | Must quantize client-side |

### The Killer Feature: Contextualized Embeddings

When you chunk a PDF and embed each chunk independently, a chunk like *"The results are shown in Table 2"* embeds as a generic reference. With contextualized embeddings, the model knows what Table 2 contains because it saw the surrounding chunks — yielding the **"golden chunk"** that actually answers the user's question.

### When to Use What

- **Use Perplexity pplx-embed** when you have structured documents (PDFs, articles, reports) where inter-chunk context matters
- **Use OpenAI text-embedding-3-large** for general-purpose embedding of independent text snippets (e.g., product descriptions, FAQ entries)

## Clean Up (Optional)

Delete the index when you're done experimenting.

In [17]:
# Uncomment to delete the index
# index_client.delete_index(INDEX_NAME)
# print(f"Index '{INDEX_NAME}' deleted")